# Session 7 · Clean and chunk a messy document

**From Prompt to Production · EvisionC**

1. Set **ROUTE** in the first cell to your model route from Session 2 (`claude`, `gemini`, `openai` or `openrouter`).
2. Add your API key in Colab Secrets (key icon, left sidebar, Notebook access on) under the name for your route: `ANTHROPIC_API_KEY`, `GEMINI_API_KEY`, `OPENAI_API_KEY` or `OPENROUTER_API_KEY`.
3. Run the cells from top to bottom (Shift+Enter). **Lumo track:** everything runs as is. **Own case:** the comments say what to swap.

The guided player explains every cell.

In [ ]:
# Cell 1 · setup (the same in every lab)
!git clone -q --depth 1 https://github.com/lrebelfor/lumo-sandbox /content/lumo 2>/dev/null || git -C /content/lumo pull -q
!pip install -q anthropic openai
import sys; sys.path.insert(0, "/content/lumo")
import lumo
from course_llm import LLM

ROUTE = "claude"   # your model route from Session 2: "claude", "openai", "gemini", "openrouter" (or "ollama" outside Colab)
llm = LLM(ROUTE)
print("Connected:", llm)

In [ ]:
import re

# The Lumo handbook as a messy text export: page headers, sentences broken over lines, double spaces.
raw = lumo.handbook_messy()          # own case: raw = open("your-export.txt").read()
print(raw[:600], "\n...")

def clean(text):
    text = re.sub(r"LUMO CUSTOMER HANDBOOK\s+v[\d.]+\s+Page \d+\n", "", text)   # remove page headers
    text = re.sub(r"[ \t]+", " ", text)                                        # squeeze repeated spaces
    text = re.sub(r"(?<![.:?!\"])\n(?=[a-z(])", " ", text)                     # rejoin broken sentences
    return text.strip()

cleaned = clean(raw)
print("\n=== CLEANED ===\n", cleaned[:600])

In [ ]:
# Fixed-size chunks with overlap. Look at the borders: words and rules get cut.
def chunk(text, size=300, overlap=60, source="handbook-export"):
    chunks, start = [], 0
    while start < len(text):
        chunks.append({"id": f"{source}-{len(chunks)}", "text": text[start:start + size],
                       "metadata": {"source": source, "start": start}})
        if start + size >= len(text):
            break
        start += size - overlap
    return chunks

chunks = chunk(cleaned)              # try size=150 and size=600: what changes?
print(len(chunks), "chunks")
for c in chunks[:4]:
    print(c["id"], c["metadata"], "\n   ", c["text"].replace("\n", " | "), "\n")

In [ ]:
# Better: keep the structure. The Markdown version of the handbook has a heading per section.
section_chunks = []
for name, text in lumo.handbook():   # own case: your own documents with # headings
    title = text.splitlines()[0].lstrip("# ")
    section_chunks.append({"id": name.removesuffix(".md"), "text": text,
                           "metadata": {"source": name, "section": title}})

# Let the model add one useful piece of metadata: the questions each chunk answers.
for c in section_chunks[:3]:
    q = llm.ask("List 3 short customer questions this text answers, one per line, nothing else:\n\n" + c["text"]).text
    c["metadata"]["answers"] = q
    print(c["id"], "->", c["metadata"]["section"], "\n", q, "\n")